[Lab README](README.md)

# Lab 4.1: The write, and the rule that stops it

Lab 3 blocked a 15-guest booking with `MaxGuestsHook`. That worked. The hook
fires before the tool runs, the model receives a cancellation instead of a
result, and no amount of rephrasing gets around it. As a demonstration that a
guardrail can sit outside the model's reach, it is exactly right.

The number is the problem. `10` is a Python literal inside a hook class, in one
notebook, next to one agent. The moment a second caller needs the same limit,
there is nothing to read it from, and the two copies start drifting the day
someone changes one of them. A business rule is connected data: it belongs where
the hotels and reservations already live, enforced inside the same boundary as
the write it governs.

This lab moves it there. The limit comes out of a `Rule` node in your graph, the
reservation command reads it inside the same transaction as the write, and the
same 15-guest request gets rejected without anything being written.

## Who owns what

| Neo4j owns | AWS owns |
|---|---|
| The maximum-guests rule, as a `Rule` node your command reads | Amazon Bedrock reasons over the retrieved evidence |
| The idempotent `ReservationRequest` write and its `FOR_HOTEL` link | |
| The uniqueness constraints that make a retry safe | |

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

print("Environment ready")

## 1. Connect, and confirm the rule is in the graph

The reservation command needs three things from your graph: the fixture hotel
IDs, three uniqueness constraints, and the `Rule` node. Lab 1 seeded all of
them. The cell below applies them again, which is idempotent, then reports
anything still missing.

In [ ]:
import json
import os
import uuid
from datetime import date, timedelta

import boto3
from dotenv import load_dotenv
from neo4j import GraphDatabase

# Safe at the top of the cell: bedrock_providers opens no client and reads
# no required variable at import. The workshop imports further down are
# inside the guard because graph_connection raises without NEO4J_PASSWORD.
from workshop.bedrock_providers import default_model_id

load_dotenv()

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
AGENT_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
# One definition, in workshop/src/workshop/bedrock_providers.py. A MODEL_ID
# in the environment still overrides it.
MODEL_ID = default_model_id()

if not NEO4J_READY:
    print("Neo4j is not configured; the live cells below will skip.")
if not BEDROCK_READY:
    print("AWS credentials are not configured; the agent cells below will skip.")

if NEO4J_READY:
    from workshop.contracts import MAX_GUESTS, MAX_GUESTS_RULE_ID, OVER_LIMIT_GUESTS
    from workshop.graph_setup import (
        HERO_SOURCE,
        RULE_QUERY,
        apply_lab4_fixtures,
        load_manifest,
        readiness_problems,
    )
    from workshop.hybrid_retrieval import Neo4jConfig
    from workshop.reservation_command import create_reservation_request

    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    # notifications_min_severity="OFF" keeps Neo4j's 01N51 and 01N52 planner
    # notices out of the output. The shared package sets it on its drivers too.
    driver = GraphDatabase.driver(
        config.uri,
        auth=(config.username, config.password),
        notifications_min_severity="OFF",
    )
    driver.verify_connectivity()

    problems = apply_lab4_fixtures(driver, config.database, manifest)
    if not problems:
        problems = readiness_problems(driver, config.database, manifest)
    if problems:
        raise RuntimeError(
            "The graph is not ready: "
            + "; ".join(problems)
            + "\nRe-run 01-graph-build/1.1_build_graph.ipynb."
        )
    print("Graph ready: fixtures applied, constraints present, rule seeded.")

### Where the limit lives now

Read it before writing anything. This is the whole argument of the lab in one
cell: the number is not in this notebook, not in a prompt, and not in a hook
class. It is a property on a node, with a rejection message and an enable flag
next to it, and any caller that can reach the graph can read the same value.

In [ ]:
if not NEO4J_READY:
    print("Skipping: Neo4j is not configured.")
else:
    with driver.session(database=config.database) as session:
        rule = session.run(RULE_QUERY, rule_id=MAX_GUESTS_RULE_ID).single()

    print(f"Rule node:  {MAX_GUESTS_RULE_ID}")
    print(f"  max_guests:        {rule['max_guests']}")
    print(f"  enabled:           {rule['enabled']}")
    print(f"  rejection_message: {rule['rejection_message']}")
    print(f"  steering_message:  {rule['steering_message']}")

    # The Python constant and the graph must agree. When they do not, the graph
    # is the one that decides, because it is what the command reads.
    assert rule["max_guests"] == MAX_GUESTS
    print(f"\nPython constant MAX_GUESTS={MAX_GUESTS} agrees with the graph.")

## 2. Register the write on `hotel_agent`

Lab 3 finished with `hotel_agent`: the hybrid retriever behind a `@tool`, and a
guest limit enforced by a hook. Rebuild it here with three changes. The write
tool `create_reservation_request` joins the toolset. `MaxGuestsHook` comes off,
because the rule it enforced now lives in the graph and the command reads it.
And the system prompt gains one instruction: pass the caller's identifiers
through unchanged, and report the command's response rather than composing one.

The tool signature is exactly the five fields of the reservation contract.
`request_id` is created by the caller, not the model, and reusing it is what
makes a retry safe.

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS, search_hotel_knowledge

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Look up amenities, ratings, and policies for a specific named hotel."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    @tool
    def create_reservation_request_tool(
        request_id: str,
        hotel_id: str,
        check_in: str,
        check_out: str,
        guests: int,
    ) -> str:
        """Create a reservation request. Reuse request_id when retrying."""
        write_driver = GraphDatabase.driver(
            config.uri,
            auth=(config.username, config.password),
            notifications_min_severity="OFF",
        )
        try:
            response = create_reservation_request(
                {
                    "request_id": request_id,
                    "hotel_id": hotel_id,
                    "check_in": check_in,
                    "check_out": check_out,
                    "guests": guests,
                },
                driver=write_driver,
                database=config.database,
            )
        finally:
            write_driver.close()
        return json.dumps(response, ensure_ascii=False)

    hotel_agent = Agent(
        name="hotel_agent",
        model=BedrockModel(model_id=MODEL_ID, region_name=AWS_REGION),
        tools=[search_hotel_knowledge_tool, create_reservation_request_tool],
        system_prompt=(
            "You are a hotel assistant. Look up hotels with "
            "search_hotel_knowledge_tool and take bookings only with "
            "create_reservation_request_tool. Pass through the request_id, "
            "hotel_id, and dates the user gives you exactly as written. Report "
            "the command's response as it comes back; never claim a booking "
            "succeeded unless the response says accepted.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print("hotel_agent now carries the retriever and the reservation write.")

## 3. A 15-guest request is rejected, and nothing is written

The dates are computed from today, so this example never expires. Two
`request_id` values are created here, one per case, because the two cases make
different points. The over-limit id shows that a rejected request leaves nothing
behind. The booking id, used in section 4, shows that the same id delivered
twice produces one node.

In [ ]:
if not NEO4J_READY:
    print("Skipping: Neo4j is not configured.")
else:
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    OVER_LIMIT_REQUEST_ID = str(uuid.uuid4())
    BOOKING_REQUEST_ID = str(uuid.uuid4())

    print(f"Hero hotel_id (from fixture manifest):    {hero_id}")
    print(f"Over-limit request_id (never written):    {OVER_LIMIT_REQUEST_ID}")
    print(f"Booking request_id (reused on retries):   {BOOKING_REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}")

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    # Strands prints the streamed response itself, through the default
    # PrintingCallbackHandler. Wrapping this call in print() would show the
    # whole answer a second time.
    hotel_agent(
        f"Book {OVER_LIMIT_GUESTS} guests into hotel_id {hero_id} from "
        f"{check_in} to {check_out}. Use request_id {OVER_LIMIT_REQUEST_ID}."
    )

The agent tried. The command refused. Nothing in the prompt told the model about
a limit of 10, so it could not have been persuaded out of one, and the rejection
came back as a structured response rather than as an apology it composed.

Here is that response on its own, called directly, so you can read the fields the
agent was working from. The same payload goes in twice. The second delivery
comes back with `duplicate` still false, which is the argument that matters: a
duplicate can only be reported against a stored request, and the rejection
stored nothing.

In [ ]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    over_limit_payload = {
        "request_id": OVER_LIMIT_REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    rejected = create_reservation_request(
        over_limit_payload, driver=driver, database=config.database
    )
    rejected_again = create_reservation_request(
        over_limit_payload, driver=driver, database=config.database
    )
    print("First delivery:")
    print(json.dumps(rejected, indent=2))
    print("\nSame request_id re-delivered:")
    print(f"  status:    {rejected_again['status']}")
    print(f"  duplicate: {rejected_again['duplicate']}")
    print("\nduplicate is false because the first delivery wrote nothing to find.")

    assert rejected["status"] == "rejected"
    assert rejected["reason_code"] == "max_guests_exceeded"
    assert rejected_again["duplicate"] is False

## 4. A valid request is recorded, and safe to retry

The booking `request_id` this time, and a party size within the limit. The
command creates one `ReservationRequest` linked to the hero hotel by
`FOR_HOTEL`. Re-delivering the identical request returns the existing record
with `duplicate=true` and creates no second node, so a retried or replayed
reservation is safe.

The proof is in the two responses side by side. Apart from the `duplicate` flag
and the message that follows from it, they are the same record, down to the same
`created_at`, so the second delivery read the first one back rather than writing
its own.

That guarantee is a uniqueness constraint in the graph, not a check the caller
remembered to write. It holds whether the retry comes from this notebook, the
agent, or a Lambda that timed out and was retried by its caller.

In [ ]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": BOOKING_REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    accepted = create_reservation_request(
        valid_payload, driver=driver, database=config.database
    )
    replay = create_reservation_request(
        valid_payload, driver=driver, database=config.database
    )
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    # The two responses describe one record. Strip the duplicate flag and the
    # message it drives, and what is left has to be identical, character for
    # character, including the Neo4j-generated created_at.
    record = {"status", "request_id", "hotel_id", "created_at"}
    first_record = json.dumps({k: accepted[k] for k in sorted(record)})
    second_record = json.dumps({k: replay[k] for k in sorted(record)})
    print(f"\nSame record both times: {first_record == second_record}")
    print(f"  {first_record}")

    assert accepted["status"] == "accepted" and not accepted["duplicate"]
    assert replay["duplicate"] is True
    assert first_record == second_record

## 5. A hotel that does not exist is rejected

The rule is one of two things the command checks. The other is identity: a
`hotel_id` that matches no `Hotel` node cannot be booked, and the command says
so with a reason code rather than creating an orphan request.

This is why grounded retrieval returns an opaque `hotel_id` and not a display
name. A name a model half-remembers will fail this check. A name it invents
outright would too.

In [ ]:
if not AGENT_READY:
    print("Skipping: needs both Neo4j and AWS credentials.")
else:
    # As above, Strands prints the streamed response; no print() wrapper here.
    hotel_agent(
        f"Book 2 guests into hotel_id hotel-does-not-exist from "
        f"{check_in} to {check_out}. Use request_id {uuid.uuid4()}."
    )

In [ ]:
if not NEO4J_READY:
    print("Skipping unknown-hotel rejection: Neo4j is not configured.")
else:
    unknown_payload = {
        "request_id": str(uuid.uuid4()),
        "hotel_id": "hotel-does-not-exist",
        "check_in": check_in,
        "check_out": check_out,
        "guests": 2,
    }
    unknown = create_reservation_request(
        unknown_payload, driver=driver, database=config.database
    )
    print(json.dumps(unknown, indent=2))
    assert unknown["status"] == "rejected"
    assert unknown["reason_code"] == "unknown_hotel"

## 6. Inspect the reservation in your graph

Anchored on the stable booking `request_id`, confirm exactly one accepted
request linked to exactly one hotel. One row, from the two deliveries of that
id. The over-limit id was delivered twice as well, three times if the agent cell
ran, and left no row at all.

In [ ]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    with driver.session(database=config.database) as session:
        rows = [dict(record) for record in session.run(query, rid=BOOKING_REQUEST_ID)]
    for row in rows:
        print(row)
    assert len(rows) == 1, f"expected exactly one request, found {len(rows)}"

## What you built

| | Where it lives | What it means |
|---|---|---|
| The guest limit | A `Rule` node in Neo4j | One value, readable by every caller, changed in one place |
| The rejection | Inside the write transaction | An over-limit request cannot be written, not even by a retry |
| Hotel identity | A `hotel_id` from grounded retrieval | An invented or half-remembered name fails the check |
| Retry safety | A uniqueness constraint | The second delivery returns the first record, not a second node |

Everything above ran on your own Aura instance and Amazon Bedrock, and created
no AWS resources.

**Next:** Lab 5 takes this same retrieval tool and this same reservation command
and runs them as a managed service on Amazon Bedrock AgentCore. The contracts do
not change. The trust boundary does.